# 09 — F1 reorder alerts

Compute reorder_point and suggested_qty per SKU × DC using organic run rate + lead time + elasticity-driven safety stock. Output alert list.

**Upstream:**
- `clean_demand_weekly.parquet` — per-week base-unit qty per (SKU × channel × DC). Used to pool across channels and compute a DC-level run rate + std.
- `inv_snapshot.parquet` — today's `Available` and `On Hand` per (SKU × DC). `Available = On Hand − Allocated`.
- `item_master.parquet` — Lead Time (freeform string, needs parsing), Case Pack, Country of Origin, Manufacturer.

**Core math** (per SKU × DC):
- `run_rate_wk = mean(weekly qty summed across channels)`
- `std_wk     = std(weekly qty summed across channels)`
- `lead_time_wk = parse(item_master.Lead Time) || 13 wk default` (upper bound on ranges, bare number = months)
- `safety_stock = 1.65 × std_wk × sqrt(lead_time_wk)` (classic 95% service level under roughly-normal demand)
- `reorder_point = run_rate_wk × lead_time_wk + safety_stock`
- `reorder_flag = available_now < reorder_point`
- `suggested_qty = max(0, reorder_point + 4·run_rate_wk − available_now)` rounded up to case pack.

**Confidence**: `high` if ≥ 8 clean weeks AND lead-time was parsed AND inv `Available` not null. `medium` if 2 of 3. `low` otherwise.

**Output:**
- `reorder_alerts.parquet` — full table, one row per (SKU × DC).
- `reorder_alerts.csv` — same data for hand-off to the UI.

**Deliberate omissions (v1):**
- No elasticity-based safety stock (used CV-derived std instead — see notes/status.md).
- MOQ is not enforced (too messy: "2,000 LB", "Ocean container size"). Shown as a note column.
- Lead-time variance from PO history is a separate next-feature, not v1.

**Promotes to:** `src/reorder.py`.

## 1. Imports

In [10]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
ART = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.reorder import (
    SERVICE_LEVEL_Z,
    FORWARD_COVER_WEEKS,
    DEFAULT_LEAD_WEEKS,
    MIN_WEEKS_FOR_HIGH,
    MIN_POS_FOR_LEAD,
    build_reorder_alerts,
    compute_dc_stats,
    compute_lead_time_from_po,
    parse_case_pack,
    parse_lead_time_weeks,
    prepare_item_master,
)

DC_MAP = {'1': 'SF', '2': 'NJ', '3': 'LA'}

## 2. Load upstream

In [11]:
weekly = pd.read_parquet(ART / 'clean_demand_weekly.parquet')
inv    = pd.read_parquet(ART / 'inv_snapshot.parquet')
im     = pd.read_parquet(ART / 'item_master.parquet')
po     = pd.read_parquet(ART / 'po.parquet')

print(f'clean_demand_weekly : {weekly.shape}  '
      f'({weekly["ITEMNMBR"].nunique()} SKU × {weekly["DC"].nunique()} DC × '
      f'{weekly["week_start"].nunique()} weeks)')
print(f'inv_snapshot        : {inv.shape}')
print(f'item_master         : {im.shape}')
print(f'po                  : {po.shape}')

clean_demand_weekly : (35591, 8)  (83 SKU × 3 DC × 172 weeks)
inv_snapshot        : (219, 5)
item_master         : (65, 15)
po                  : (5281, 16)


## 3. Do the work

In [12]:
# Sanity-peek the upstream lookups that feed the alert builder before we
# collapse it all into one call. Useful for buyer-facing "where did that
# come from" questions and for diagnosing low-confidence rows later.
im_p = prepare_item_master(im)
print('Lead-time parse sample (item master):')
print(im_p[['ITEMNMBR', 'Lead Time', 'lead_time_wk', 'lead_parsed']].head(10).to_string())
print(f'\nparsed ok : {int(im_p["lead_parsed"].sum())} / {len(im_p)}')
print(f'null      : {int(im_p["Lead Time"].isna().sum())}')

po_lead = compute_lead_time_from_po(po, DC_MAP)
# compute_lead_time_from_po now returns (per_dc, per_sku_pool) — unpack if tuple.
if isinstance(po_lead, tuple):
    po_lead_dc, _po_lead_pool = po_lead
else:
    po_lead_dc = po_lead
print(f'\nPO-history lead coverage (≥{MIN_POS_FOR_LEAD} POs): {len(po_lead_dc)} lanes')
print(po_lead_dc[po_lead_dc['ITEMNMBR'] == 'T-32206'][['ITEMNMBR', 'DC', 'lead_time_days_med', 'n_pos', 'lead_time_wk_po']].to_string())

dc_stats = compute_dc_stats(weekly)
print(f'\ndc_stats shape : {dc_stats.shape}')

# Best-policy call: trend-aware regime detection (window-tightening for
# declining/growing lanes) + empirical-p99 safety stock + 70% hybrid floor
# against the legacy baseline. Per 10c verdict: WMAPE 62% -> 32%, bias +41% -> +6%.
alerts = build_reorder_alerts(
    weekly, inv, im, po=po, dc_map=DC_MAP,
    use_trend_regime=True,
    use_empirical_p99_ss=True,
    ss_floor_frac=0.70,
)
print(f'\nreorder_alerts: {alerts.shape}')
print(f'rows flagged for reorder : {int(alerts["reorder_flag"].sum())} / {len(alerts)}')
print(f'confidence counts        : {alerts["confidence"].value_counts().to_dict()}')
print(f'lead_time_source         : {alerts["lead_time_source"].value_counts().to_dict()}')
if 'regime' in alerts.columns:
    print(f'regime counts            : {alerts["regime"].value_counts(dropna=False).to_dict()}')
alerts.head(10)

Lead-time parse sample (item master):
   ITEMNMBR Lead Time  lead_time_wk  lead_parsed
0   A-61220       NaN           NaN        False
1   A-61243       NaN           NaN        False
2  A-61260N       NaN           NaN        False
3   A-61280       NaN           NaN        False
4   A-61012  3 months         12.99         True
5  AC-B16BK       NaN           NaN        False
6   AC-B4BK       NaN           NaN        False
7   AC-B8BK       NaN           NaN        False
8  AC-B3SLJ       NaN           NaN        False
9  AC-B6SLJ       NaN           NaN        False

parsed ok : 51 / 65
null      : 14

PO-history lead coverage (≥3 POs): 156 lanes
    ITEMNMBR  DC  lead_time_days_med  n_pos  lead_time_wk_po
147  T-32206  LA                19.0     23         2.714286
148  T-32206  NJ                36.0     51         5.142857
149  T-32206  SF                20.0     30         2.857143

dc_stats shape : (233, 9)

reorder_alerts: (233, 42)
rows flagged for reorder : 176 / 233
confid

,ITEMNMBR,Description,DC,vendor,country,run_rate_wk,std_wk,cv,n_clean_weeks,abc,...,last_week,confidence,regime,trend_ratio,n_weeks_effective,run_rate_wk_expanding,std_wk_expanding,p99_lt,ss_baseline,ss_floor
0,A-61117,AM GSG Candy 1 lb / Bag,NJ,GL,China,1005.613793,996.174747,0.994047,145,B,...,2025-12-29,high,stable,0.827700,145,1005.613793,999.627703,23669.59,16824.566361,11777.196453
1,A-61280,Am GSG Rt Tea 80 Bags (US Costco),SF,ST,USA,1309.384615,1147.254484,0.911955,13,C,...,2025-06-16,medium,stable,1.000000,13,1309.384615,1194.100326,17022.00,5510.899143,3857.629400
2,AC-B9SL,POP AM GSG Root Slices 9 oz,SF,POP,USA,20.000000,10.839742,0.625833,4,C,...,2025-05-05,low,stable,1.000000,4,20.000000,12.516656,0.00,57.765688,40.435981
3,AC-B9SLE,NaN,SF,NaN,NaN,2320.360000,657.030586,0.288998,25,A,...,2025-09-29,medium,stable,1.000000,25,2320.360000,670.579033,31159.00,4738.901895,3317.231326
4,F-04220,POP Ginger Chews Original 2 oz (24ct/cs),NJ,SNA,Indonesia,1553.633333,1387.885021,0.908587,30,B,...,2025-12-29,high,stable,1.052743,30,1553.633333,1411.611270,27446.70,8394.670163,5876.269114
5,F-04220,POP Ginger Chews Original 2 oz (24ct/cs),SF,SNA,Indonesia,619.444444,394.171737,0.654779,18,B,...,2025-12-22,high,stable,1.000000,18,619.444444,405.599371,9566.85,2412.047147,1688.433003
6,F-04221,POP Ginger Chews Lemon 2 oz (24ct/cs),NJ,SNA,Indonesia,1669.413793,1510.321421,0.920715,29,B,...,2025-12-29,high,stable,1.027514,29,1669.413793,1537.054849,30549.12,9140.666946,6398.466862
7,F-04221,POP Ginger Chews Lemon 2 oz (24ct/cs),SF,SNA,Indonesia,774.687500,547.707235,0.730191,16,B,...,2025-12-22,high,stable,1.000000,16,774.687500,565.669600,11332.94,3363.964152,2354.774906
8,F-12418,Ferrero 24 Pcs Floorstand (48),LA,Ferrero USA,Poland,12.457143,15.149433,1.233879,35,B,...,2026-01-12,high,stable,1.142378,35,12.457143,15.370604,100.60,41.628315,29.139821
9,F-12998,Ferrero Rocher 48's Flat Gift Box,LA,Ferrero USA,Poland,143.384615,249.986319,2.038992,93,C,...,2026-03-16,high,growing,1.314547,26,109.075269,222.403581,1955.20,389.613840,272.729688


## 4. Validate

In [13]:
# ---- Shape / null checks -----------------------------------------------------
assert len(alerts) == alerts[['ITEMNMBR', 'DC']].drop_duplicates().shape[0], \
    'duplicate (ITEMNMBR, DC) rows'
assert alerts['reorder_point'].notna().all(), 'null reorder_point'
assert (alerts['safety_stock'] >= 0).all(), 'negative safety_stock'

print('=== Top 15 reorder-flagged rows by weeks_of_cover ===')
cols = ['ITEMNMBR', 'Description', 'DC', 'available_now', 'run_rate_wk',
        'weeks_of_cover', 'reorder_point', 'suggested_qty', 'confidence']
print(alerts[alerts['reorder_flag']].head(15)[cols].to_string(index=False))

# ---- Reorder-flag distribution by DC ---------------------------------------
print('\n=== reorder_flag by DC ===')
print(alerts.groupby('DC')['reorder_flag'].agg(['sum', 'count']).to_string())

print('\n=== confidence by flag ===')
print(alerts.groupby(['reorder_flag', 'confidence']).size().unstack(fill_value=0).to_string())

# ---- T-32206 spot check ----------------------------------------------------
print('\n=== T-32206 (Tiger Balm Patch Warm) — showcase SKU ===')
spot = alerts[alerts['ITEMNMBR'] == 'T-32206'][cols + [
    'lead_time_wk', 'lead_time_source', 'std_wk', 'safety_stock',
    'case_pack', 'suggested_cases',
]]
print(spot.to_string(index=False))

# SF is our known stockout lane (the pipeline earlier showed on_hand dipping to
# -17k base units in 2023). Sanity-check the numbers:
#   - Lead Time in item_master is "Half a year or more" -> 26 weeks (parsed)
#   - run_rate should land near 4k-15k/week per DC (we saw that in step 07)
#   - safety_stock should be a meaningful fraction of run_rate * lead_time
t_sf = alerts[(alerts['ITEMNMBR'] == 'T-32206') & (alerts['DC'] == 'SF')].iloc[0]
print(f'\n--- T-32206 SF manual trace ---')
print(f'  run_rate_wk     : {t_sf["run_rate_wk"]:>10,.1f} u/wk')
print(f'  std_wk          : {t_sf["std_wk"]:>10,.1f} u/wk')
print(f'  lead_time_wk    : {t_sf["lead_time_wk"]:>10,.1f}  ({t_sf["lead_time_source"]})')
print(f'  demand@lead     : {t_sf["run_rate_wk"]*t_sf["lead_time_wk"]:>10,.0f} u')
print(f'  safety_stock    : {t_sf["safety_stock"]:>10,.0f} u  (= 1.65 * std * sqrt(lead))')
print(f'  reorder_point   : {t_sf["reorder_point"]:>10,.0f} u')
print(f'  available_now   : {t_sf["available_now"]:>10,.0f} u')
print(f'  weeks_of_cover  : {t_sf["weeks_of_cover"]:>10,.2f} wk')
print(f'  reorder_flag    : {bool(t_sf["reorder_flag"])}')
print(f'  suggested_qty   : {t_sf["suggested_qty"]:>10,.0f} u '
      f'({t_sf["suggested_cases"]:.1f} cases of {int(t_sf["case_pack"])})')

=== Top 15 reorder-flagged rows by weeks_of_cover ===
ITEMNMBR                              Description DC  available_now  run_rate_wk  weeks_of_cover  reorder_point  suggested_qty confidence
 A-61117                  AM GSG Candy 1 lb / Bag NJ            0.0  1005.613793        0.000000   3.319262e+04   3.924800e+04       high
 A-61280        Am GSG Rt Tea 80 Bags (US Costco) SF            0.0  1309.384615        0.000000   2.231670e+04   3.017301e+04     medium
 AC-B9SL              POP AM GSG Root Slices 9 oz SF            0.0    20.000000        0.000000   3.100266e+02   4.800000e+02        low
AC-B9SLE                                      NaN SF            0.0  2320.360000        0.000000   3.480784e+04   4.873000e+04     medium
 F-04220 POP Ginger Chews Original 2 oz (24ct/cs) NJ            0.0  1553.633333        0.000000   2.843527e+04   3.775707e+04       high
 F-04220 POP Ginger Chews Original 2 oz (24ct/cs) SF            0.0   619.444444        0.000000   1.039067e+04   1.41

## 5. Save downstream artifact

In [14]:
alerts.to_parquet(ART / 'reorder_alerts.parquet')
print(f'wrote parquet: {ART / "reorder_alerts.parquet"}  ({alerts.shape})')

# CSV hand-off for the UI. Explicit encoding + quoting rules so the
# buyer-facing consumer can parse without surprises.
csv_path = ART / 'reorder_alerts.csv'
alerts.to_csv(csv_path, index=False)
print(f'wrote csv    : {csv_path}')

wrote parquet: /Users/johnpork/repos/pop_prompt2/pipeline/artifacts/reorder_alerts.parquet  ((233, 42))
wrote csv    : /Users/johnpork/repos/pop_prompt2/pipeline/artifacts/reorder_alerts.csv


## 6. Promote

Once validation above looks right, extract the core logic into `src/reorder.py` and replace the inline code here with `from src.<module> import ...`. Downstream dev notebooks can then import the same function.

## 7. Export to UI

Dump this notebook's artifact as `ui/data/alerts_today.json` for the reorder-alert UI. Embeds a 26-week `on_hand_sparkline` per row so the List view can render inline sparklines without a separate fetch.

In [15]:
import json
from pathlib import Path
import pandas as pd

UI_DATA = Path('../ui/data')
UI_DATA.mkdir(parents=True, exist_ok=True)

inv_weekly = pd.read_parquet('artifacts/inv_weekly.parquet')

SPARK_WEEKS = 26
latest = inv_weekly['week_start'].max()
cutoff = latest - pd.Timedelta(weeks=SPARK_WEEKS - 1)
spark_src = (
    inv_weekly[inv_weekly['week_start'] >= cutoff]
    .sort_values(['ITEMNMBR', 'DC', 'week_start'])
    .groupby(['ITEMNMBR', 'DC'])['on_hand_est']
    .apply(lambda s: [None if pd.isna(v) else float(v) for v in s.tolist()])
    .rename('on_hand_sparkline')
    .reset_index()
)

reorder_alerts = alerts
out = reorder_alerts.merge(spark_src, on=['ITEMNMBR', 'DC'], how='left')
out['on_hand_sparkline'] = out['on_hand_sparkline'].apply(
    lambda v: v if isinstance(v, list) else []
)

payload = out.replace({pd.NA: None}).where(pd.notna(out), None).to_dict(orient='records')
for row in payload:
    for k, v in list(row.items()):
        if isinstance(v, float) and (v != v):
            row[k] = None

(UI_DATA / 'alerts_today.json').write_text(json.dumps(payload, default=str))
print(f"wrote {len(payload)} rows to {UI_DATA / 'alerts_today.json'}")

wrote 233 rows to ../ui/data/alerts_today.json
